# Module 10: Cluster Computing — Running PyAutoLens on Slurm

## Learning to Autolens

---

**Purpose:** So far, every module has run on your laptop. That's fine for *teaching* — but when we tried to run the full Module 04 SLaM pipeline locally, `search_2` stalled for 15+ hours and we had to kill it. Real science runs on a cluster. This module shows how to take any tutorial notebook (using Module 04 as the worked example) and turn it into a Slurm job on Harvard's **Cannon** cluster (FAS Research Computing).

**Prerequisites:**
- Module 03 and Module 04 conceptually (we'll reuse Module 04's code, not its notebook).
- A Cannon account. If you don't have one, apply via <https://www.rc.fas.harvard.edu/> — Harvard CfA affiliates qualify for shared partitions for free.
- Comfortable with `ssh`, `rsync`, and reading job logs.

**What you'll build:** a reusable pattern — `fit_*.py` + `submit_*.slurm` + `push/pull` rsync — that converts any notebook into a cluster job.

---

## Table of Contents

1. [Why Move to a Cluster?](#1-why-cluster)
2. [The Three-Part Pattern](#2-pattern)
3. [Part A: Extract the Fit as a Standalone Script](#3-extract)
4. [Part B: Write the Slurm Submission Script](#4-slurm)
5. [Part C: Transfer Data with rsync](#5-rsync)
6. [Nautilus Checkpoint Resume](#6-resume)
7. [Worked Example: Module 04 on Cannon](#7-example)
8. [Monitoring and Debugging](#8-monitoring)
9. [FASRC-Specific Notes](#9-fasrc)
10. [Exercises](#10-exercises)

---

## 1. Why Move to a Cluster? <a id="1-why-cluster"></a>

### The symptom

On a 2023 MacBook Pro (M2 Max, 12 cores), running Module 04 end-to-end takes:

| Stage | Parameters | Local wall-clock | What happened |
|-------|-----------|------------------|----------------|
| `chain/search_1` (SIS) | 3 | ~35 min | fine |
| `chain/search_2` (SIE + shear) | ~14 | **stalled > 15 h** | Nautilus bounds = 203 for hours |
| SLaM SOURCE LP | ~20 | ~2 h | fine |
| SLaM SOURCE PIX × 2 | — | ~3 h | fine |
| SLaM LIGHT LP | ~6 | ~1 h | fine |
| SLaM MASS TOTAL | ~7 | ~1.5 h | fine |

So the total is ~24 h of successful work plus one indefinitely-stuck search. On a cluster with 32+ cores, the same pipeline typically completes in 4–8 h and — critically — you can **walk away** from it.

### Three things clusters buy you

1. **More cores.** Nautilus (and dynesty) parallelize likelihood evaluations. `number_of_cores=16` on Cannon routinely gives 10–12× speedups over `number_of_cores=4` on a laptop.
2. **More RAM.** Pixelized source inversions build a $N_{\rm pix} \times N_{\rm pix}$ mapping matrix. 32 GB on a laptop is tight for 4k × 4k HST cutouts; Cannon nodes offer 256–512 GB routinely.
3. **Time independence.** You don't need your laptop open and unpaused. The job runs even if you close the lid or your wifi drops. Slurm resubmits automatically if a node dies.

### What you *don't* get for free

- **No GUI.** You cannot call `aplt.subplot_fit()` in the cluster script and expect a plot to pop up. All visualization happens back on your laptop after you `rsync` the results.
- **No `get_ipython()`.** The `%matplotlib inline` magic breaks. Remove all Jupyter magics from the extracted fit script.
- **Slow filesystem.** Most HPC scratch filesystems are NFS-backed. Writing thousands of small files (like each Nautilus checkpoint step) can dominate runtime. Use `$SCRATCH`, not `$HOME`, for Nautilus output.

---

## 2. The Three-Part Pattern <a id="2-pattern"></a>

Every cluster-ready PyAutoLens job has three pieces:

```
  my_notebook.ipynb           (stays on laptop — visualization/analysis only)
         │
         ▼   extract fit code
  fit_notebook.py             (runs on cluster — no magics, CLI-driven)
         │
         ▼   wrapped by
  submit_notebook.slurm       (SBATCH directives + module loads)
         │
         ▼   data flow
  push_to_cluster.sh          (code + dataset + existing checkpoints → cluster)
  pull_from_cluster.sh        (Nautilus outputs → laptop for plotting)
```

This repo ships a worked instance of all five files for Module 04 in `Modules/10_Cluster_Computing/scripts/`:

| File | Purpose |
|------|---------|
| `fit_module04.py` | Standalone Python — replicates Module 04's two-search chain and full SLaM pipeline |
| `submit_cannon.slurm` | Slurm SBATCH script for FASRC Cannon |
| `push_to_cannon.sh` | rsync laptop → Cannon (code + data + checkpoints) |
| `pull_from_cannon.sh` | rsync Cannon → laptop (Nautilus output) |
| (implicit) | `cannon_output/` — where results land on the laptop after pulling |

---

## 3. Part A: Extract the Fit as a Standalone Script <a id="3-extract"></a>

### Start from `jupyter nbconvert`

The mechanical part:

```bash
jupyter nbconvert --to script \
    Modules/04_Search_Chaining_SLaM/04_search_chaining_slam.ipynb \
    --stdout > /tmp/mod04_raw.py
```

This gives you a `.py` file with all code cells concatenated. **Do not ship this directly** — it contains Jupyter magics, interactive plotting, and hard-coded relative paths. Use it as a starting point.

### Five edits you always need to make

1. **Strip magics and plotters.**
   ```python
   # remove these lines:
   get_ipython().run_line_magic('matplotlib', 'inline')
   dataset_plotter.subplot_dataset()
   fit_plotter.subplot_fit()
   ```
   The cluster has no display. Plots are a *post-hoc* laptop activity.

2. **Replace relative paths with CLI arguments.** The notebook uses `Path('../../autolens_workspace_original/dataset/imaging/simple')` which only resolves from the notebook's working directory. On the cluster, pass:
   ```python
   parser.add_argument('--dataset-root', type=Path, required=True)
   ```
   and let the Slurm script supply the absolute path.

3. **Wire `number_of_cores` to `$SLURM_CPUS_PER_TASK`.** PyAutoFit's `af.SettingsSearch` accepts `number_of_cores=`. On a cluster you want this to equal the cores Slurm allocated:
   ```python
   settings_search = af.SettingsSearch(
       path_prefix=...,
       unique_tag=...,
       number_of_cores=int(os.environ.get('SLURM_CPUS_PER_TASK', '1')),
   )
   ```

4. **Bump `n_live`.** The hung `search_2` on the laptop is the canonical failure mode — Nautilus with `n_live=75` can produce a numerically singular neural-bound covariance (the `numpy.linalg.cholesky: Matrix is not positive definite` crash we saw in Solution 04). On the cluster we can afford `n_live=100` to 150 for modest-dim searches and `n_live=200`+ for the full SLaM final stage, and we don't care about the extra runtime.

5. **`flush=True` on every `print`.** Slurm buffers stdout and flushes only when the buffer fills or the process exits. If the job crashes, unflushed output is *gone*. This one-liner has saved more debugging hours than any other trick:
   ```python
   print(f"[SLaM] SOURCE LP done in {dt:.1f} min", flush=True)
   ```

See `scripts/fit_module04.py` in this module for the full example.

In [ ]:
# Peek at the extracted fit script structure
!head -30 scripts/fit_module04.py

---

## 4. Part B: Write the Slurm Submission Script <a id="4-slurm"></a>

A Slurm script has two layers: **directives** (`#SBATCH` lines, parsed before the shell runs) and **body** (a regular bash script that Slurm invokes).

### Anatomy of `submit_cannon.slurm`

```bash
#!/bin/bash
#SBATCH --job-name=autolens_mod04      # shows in `squeue`
#SBATCH --partition=shared             # free-for-all Cannon partition
#SBATCH --time=24:00:00                # HH:MM:SS wall limit
#SBATCH --nodes=1                      # single-node (don't need MPI)
#SBATCH --ntasks=1                     # one Python process
#SBATCH --cpus-per-task=16             # Nautilus uses these for likelihoods
#SBATCH --mem=32G                      # RAM. 32G is comfortable for `simple` dataset
#SBATCH --output=logs/mod04_%j.out     # %j = job ID
#SBATCH --error=logs/mod04_%j.err
#SBATCH --mail-type=BEGIN,END,FAIL     # get emails
```

### Choosing partition and time

| Job profile | Partition | Time | Reasoning |
|-------------|-----------|------|-----------|
| Quick test, ≤4 h, ≤8 cores | `test` or `shared` | `4:00:00` | Fast queue |
| Full SLaM pipeline, 8–24 h, 16+ cores | `shared` | `24:00:00` | Usually starts in <1 h |
| Real-data SLaM (HST/JWST), 24+ h | `shared` (long) or `serial_requeue` | `72:00:00` | `serial_requeue` is interruptible but much faster to dispatch |
| Many parallel cutouts (embarrassingly parallel) | `shared` with `--array=0-49` | per-job | One Slurm job per lens |

### Module loads

FASRC uses Lmod (`module load`). On Cannon rocky8 the current working combo is:

```bash
module purge
module load python/3.11.8-fasrc01
module load Anaconda3/2024.06-fasrc01
source activate autolens          # conda env you created once with `pip install autolens`
```

Do the `pip install autolens` step **once** in an interactive session (`salloc --partition=test --time=1:00:00`), not from the submit script.

In [ ]:
# Inspect the submission script
!cat scripts/submit_cannon.slurm

---

## 5. Part C: Transfer Data with rsync <a id="5-rsync"></a>

`rsync` is the right tool: incremental, resumable, and ignores files that haven't changed. Never use `scp -r` for this — it re-copies everything every time.

### What to push

```
Learning_to_Autolens/
  ├── Modules/                 ← code
  ├── Solutions/               ← not strictly needed on cluster, but cheap
  ├── slam_v2026.py            ← REQUIRED — Module 04 imports this
  ├── autolens_workspace_original/
  │     └── dataset/           ← REQUIRED — small FITS files
  └── Modules/04_.../output/   ← INCLUDE existing checkpoints for resume
```

### What to skip

- `.git/` — keep the cluster copy decoupled from your working tree.
- `.claude/`, `.ipynb_checkpoints/`, `__pycache__/` — noise.
- `autolens_workspace_latest/` — 400 MB+, only needed for Module 09. Push separately if/when you want to run it.
- `Output/` (top-level) — will be regenerated on the cluster into `$SCRATCH`.

### Pattern

The `scripts/push_to_cannon.sh` wrapper encodes the full include/exclude list. Always do a dry run first:

```bash
./scripts/push_to_cannon.sh            # dry run — shows what *would* transfer
./scripts/push_to_cannon.sh --go       # actual transfer
```

### Pulling results back

After the job finishes (email notification, or check via `sacct -j $JOB_ID`):

```bash
./scripts/pull_from_cannon.sh --go
```

This lands Nautilus outputs in `Modules/10_Cluster_Computing/cannon_output/` on the laptop. From there you can load them in a notebook:

```python
import autofit as af
search = af.Nautilus(
    path_prefix='Modules/10_Cluster_Computing/cannon_output/module_04/slam',
    name='source_lp[1]_light[lp]_mass[total]_source[lp]',
    unique_tag='simple',
)
result = search.result  # reconstructed from the pickled samples
```

---

## 6. Nautilus Checkpoint Resume <a id="6-resume"></a>

Nautilus writes a `checkpoint.hdf5` inside each search's `files/search_internal/` directory after every bound update. If the file exists when a search starts, Nautilus **automatically** resumes from it. There is no flag to enable or disable this — it just happens.

### What this means for you

1. **Slurm hit the time limit.** Resubmit the same `sbatch submit_cannon.slurm` — no changes needed. The finished searches skip instantly; the partial one resumes.
2. **A node crashed or you cancelled the job.** Same — resubmit.
3. **You ran locally until it stalled, then moved to the cluster.** This is what we did for Module 04:
   - Local run produced `Modules/04_.../output/output/module_04/chaining/simple__no_lens_light/search_2_sie_nolenslight/.../checkpoint.hdf5` (96 MB, 203 bounds).
   - `push_to_cannon.sh` includes this file in the rsync.
   - On the cluster, Nautilus sees the checkpoint and resumes — no duplicate work.

### Checkpoint structure (for reference)

Inside each search directory on the cluster after a run:

```
search_2_sie_nolenslight/
  └── <hash>/
      ├── image/                   ← best-fit image, residual, chi^2 maps
      ├── files/
      │   ├── samples.csv          ← posterior samples
      │   ├── search_internal/
      │   │   └── checkpoint.hdf5  ← Nautilus state (resume source)
      │   └── ...
      └── info/                    ← human-readable model.info
```

### When NOT to resume

If you change the model (add a parameter, re-bound a prior) *and* reuse the same `unique_tag`, the checkpoint's dimension will not match the new model and Nautilus will error on load. The fix: change `unique_tag` (or `name`) to a fresh string — Nautilus writes into a new hashed subdir, starting fresh.

---

## 7. Worked Example: Module 04 on Cannon <a id="7-example"></a>

End-to-end, from your laptop terminal (not the notebook):

### One-time setup on Cannon

```bash
ssh rcordovarosado@login.rc.fas.harvard.edu
salloc --partition=test --time=1:00:00 --cpus-per-task=4 --mem=8G

module load python/3.11.8-fasrc01 Anaconda3/2024.06-fasrc01
conda create -n autolens python=3.11 -y
conda activate autolens
pip install --upgrade pip
pip install autolens==2026.2.26.4 numba
python -c 'import autolens as al; print(al.__version__)'
exit   # leave the interactive allocation
```

### Every submission

```bash
# from the local repo root
cd ~/Documents/AGEL/Learning_to_Autolens

# 1. Push code + data + any existing checkpoint
./Modules/10_Cluster_Computing/scripts/push_to_cannon.sh --go

# 2. Submit
ssh rcordovarosado@login.rc.fas.harvard.edu \
    "cd learning_to_autolens && sbatch Modules/10_Cluster_Computing/scripts/submit_cannon.slurm"

# 3. Wait for email (or poll: `ssh ... squeue -u rcordovarosado`)

# 4. Pull results
./Modules/10_Cluster_Computing/scripts/pull_from_cannon.sh --go
```

### Expected runtime

On Cannon `shared` partition with 16 cores / 32 GB:

| Stage | Wall time | Note |
|-------|-----------|------|
| Chain search_1 (SIS, n_live=100) | ~15 min | |
| Chain search_2 (SIE+shear, n_live=150) | ~45 min | resumes from local checkpoint |
| SLaM SOURCE LP | ~45 min | |
| SLaM SOURCE PIX run_1 | ~1 h | |
| SLaM SOURCE PIX run_2 | ~1.5 h | |
| SLaM LIGHT LP | ~30 min | |
| SLaM MASS TOTAL | ~45 min | |
| **Total** | **~5.5 h** | Comfortably under 24 h wall limit |

---

## 8. Monitoring and Debugging <a id="8-monitoring"></a>

### Commands cheat sheet

```bash
squeue -u $USER                  # all my jobs
squeue -j $JOB_ID                # one job (state, node, time used)
sacct -j $JOB_ID                 # post-mortem: exit code, max mem, CPU time
seff $JOB_ID                     # pretty efficiency report
scontrol show job $JOB_ID        # full dump

scancel $JOB_ID                  # kill it
scancel -u $USER                 # kill ALL my jobs (nuclear option)

tail -f logs/mod04_$JOB_ID.out   # live stdout (if `flush=True` is set!)
tail -f logs/mod04_$JOB_ID.err   # live stderr
```

### Common failure modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Job dies immediately with `ModuleNotFoundError: autolens` | conda env not activated | Add `source activate autolens` before the `srun` line |
| `TIMEOUT` in sacct | hit wall limit | Resubmit; Nautilus resumes automatically |
| `OOM` / `OUT_OF_MEMORY` | mem too low | Raise `--mem=` — try 64G, 128G |
| `LinAlgError: Matrix not positive definite` | `n_live` too low | Bump `n_live` by 25–50 |
| No output in `.out`, job `RUNNING` for hours | `print` without `flush=True` | Fix the script; `scancel`; resubmit |
| `FileNotFoundError: data.fits` | path arg wrong | Check `$DATASET_ROOT` in Slurm script |
| `ModuleNotFoundError: slam_v2026` | `--repo-root` wrong | Check that `slam_v2026.py` is in the pushed tree |

---

## 9. FASRC-Specific Notes <a id="9-fasrc"></a>

### Filesystems

| Path | Size | Persistence | Use for |
|------|------|-------------|---------|
| `$HOME` | 100 GB | permanent, backed up | code, small configs, conda envs |
| `$SCRATCH` (`/n/holyscratch01/...`) | 50 TB | **purged after 90 days** | Nautilus outputs, checkpoints, intermediate files |
| `/n/holylabs/LABS/<pi>/Lab` | varies | permanent, not backed up | final published results |

Rule of thumb: code in `$HOME`, scratch-heavy I/O to `$SCRATCH`, archived results to `Lab/`.

**Do not** write Nautilus outputs to `$HOME` — the quota is too small and the I/O is slow.

### Partitions (as of 2026)

| Partition | Max time | Cores/node | Queue speed | Notes |
|-----------|----------|------------|-------------|-------|
| `test` | 8 h | up to 8 | seconds | Debugging |
| `shared` | 7 d | up to 48 | minutes | Workhorse |
| `serial_requeue` | 7 d | up to 48 | seconds | **Can be killed and requeued** — fine for checkpointed work |
| `gpu` | 7 d | 32 + A100 | minutes | Not currently used by PyAutoLens |

`serial_requeue` is the hidden gem for Nautilus jobs: it dispatches fast because it volunteers to be killed, and since we have checkpoint resume, killing-and-requeuing is basically free. Use it once you trust your checkpoint setup.

### SSH config

Add this to `~/.ssh/config` locally to avoid typing your username and host every time:

```
Host cannon
    HostName login.rc.fas.harvard.edu
    User rcordovarosado
    ServerAliveInterval 60
    ControlMaster auto
    ControlPath ~/.ssh/cm-%r@%h:%p
    ControlPersist 10m
```

Then: `ssh cannon`, `rsync -a ... cannon:~/...`, etc. The `ControlMaster` block reuses one SSH connection across rsync calls — dramatic speedup.

### Two-factor

Cannon requires Duo 2FA on every login unless you have an SSH key + Kerberos. For frequent syncing, generate a key and register it per FASRC docs; otherwise budget 20 seconds per `push_to_cannon.sh` for the Duo push.

---

## 10. Exercises <a id="10-exercises"></a>

### Exercise 1: Module 03 on the cluster

Module 03 ran fine locally, but it's a good practice target. Write `fit_module03.py` by `nbconvert`-ing Module 03, following the five edits in Section 3. Write `submit_module03.slurm` (can copy from `submit_cannon.slurm` with a new `--job-name`). Submit, pull results, and confirm the corner plot matches the one you got locally.

### Exercise 2: Array job for multiple lenses

Cannon supports job arrays. Modify the Slurm script so that a single `sbatch` launches N parallel jobs, one per target, via `#SBATCH --array=0-9`. Inside the script, read `$SLURM_ARRAY_TASK_ID` and use it to index into a list of dataset names. This is the pattern you'll want when we get to Module 07 (real data: AGEL lens sample).

### Exercise 3: Measure the speedup

Time the SOURCE LP stage on your laptop and on Cannon at `--cpus-per-task=4, 8, 16, 32`. Plot wall time vs. cores. Nautilus scales sublinearly past ~16 cores because the neural-bound update is serial — identify roughly where your diminishing returns kick in.

### Exercise 4: Kill-and-resume drill

Submit a short run. After ~10 min, `scancel` it. Verify the `checkpoint.hdf5` is non-empty. Resubmit the same script unchanged. Confirm from the new job's log that Nautilus reports a non-zero starting bound count — that's proof it resumed.

---

## Summary

| Piece | File | Role |
|-------|------|------|
| Fit code | `scripts/fit_module04.py` | The notebook, minus magics and plotters, plus CLI + `flush=True` |
| Slurm | `scripts/submit_cannon.slurm` | SBATCH directives + module loads + one `srun` |
| Push | `scripts/push_to_cannon.sh` | rsync laptop → cluster (include dataset + checkpoints) |
| Pull | `scripts/pull_from_cannon.sh` | rsync cluster → laptop (Nautilus output) |
| Resume | (implicit, via `checkpoint.hdf5`) | Nautilus does this automatically |

**Next module:** Module 07 (real FITS data). Everything in this module applies directly — an AGEL lens with a 4k × 4k HST cutout is *exactly* when the cluster stops being optional.

---

*Learning to Autolens — Module 10*
*Rodrigo Córdova Rosado, Harvard CfA*
*Built with Claude Code*